# PySpark DataFrame Data Engineering Project — Hospitality Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `hotels.csv`, `hotel_bookings.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("hotels.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("hotel_bookings.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select hotel_id and city from the first DataFrame.

In [ ]:
df1.select('hotel_id','city').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where room_charge is greater than its average.

In [ ]:
avg_value = df2.select(avg('room_charge').alias('avg')).first()['avg']
df2.filter(col('room_charge') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on room_charge using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('room_charge') >= avg('room_charge').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in city and room_type with 'Unknown'.

In [ ]:
df1.fillna({'city': 'Unknown'}).show(5)
df2.fillna({'room_type': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using hotel_id and hotel_id.

In [ ]:
joined = df1.join(df2, df1['hotel_id'] == df2['hotel_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['hotel_id'] == df2['hotel_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average room_charge by room_type.

In [ ]:
df2.groupBy('room_type').agg(count('*').alias('records'), avg('room_charge').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total room_charge by room_type.

In [ ]:
df2.groupBy('room_type').agg(sum('room_charge').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum room_charge in each room_type.

In [ ]:
df2.groupBy('room_type').agg(max('room_charge').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum room_charge in each room_type.

In [ ]:
df2.groupBy('room_type').agg(min('room_charge').alias('min_value')).show()

## Q14. Having Condition

Show room_type groups whose total room_charge is greater than the overall average room_charge.

In [ ]:
overall = df2.select(avg('room_charge').alias('v')).first()['v']
df2.groupBy('room_type').agg(sum('room_charge').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by room_charge.

In [ ]:
df2.orderBy(desc('room_charge')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by room_charge.

In [ ]:
df2.orderBy(asc('room_charge')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by room_charge within each room_type.

In [ ]:
w = Window.partitionBy('room_type').orderBy(desc('room_charge'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of room_charge within each room_type.

In [ ]:
w = Window.partitionBy('room_type').orderBy('booking_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('room_charge').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's room_charge with the previous record in its room_type.

In [ ]:
w = Window.partitionBy('room_type').orderBy('booking_date')
df2.withColumn('previous_value', lag('room_charge').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each room_type's contribution to total room_charge.

In [ ]:
tot = df2.select(sum('room_charge').alias('t')).first()['t']
df2.groupBy('room_type').agg(sum('room_charge').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from booking_date.

In [ ]:
df2.withColumn('year', year('booking_date')).withColumn('month', month('booking_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average room_charge by month.

In [ ]:
df2.withColumn('month', date_format('booking_date','yyyy-MM')).groupBy('month').agg(sum('room_charge').alias('total'), avg('room_charge').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with room_type as rows and year of booking_date as columns.

In [ ]:
df2.withColumn('year', year('booking_date')).groupBy('room_type').pivot('year').agg(sum('room_charge')).show()

## Q24. Conditional KPI

Count records where room_charge is above the category average.

In [ ]:
w=Window.partitionBy('room_type')
df2.withColumn('cat_avg',avg('room_charge').over(w)).filter(col('room_charge')>col('cat_avg')).groupBy('room_type').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total room_charge by city.

In [ ]:
df1.join(df2, df1['hotel_id']==df2['hotel_id']).groupBy(df1['city']).agg(sum(df2['room_charge']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank city by total room_charge.

In [ ]:
r=df1.join(df2,df1['hotel_id']==df2['hotel_id']).groupBy(df1['city']).agg(sum(df2['room_charge']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for room_charge.

In [ ]:
df2.withColumn('segment',when(col('room_charge')<1000,'Low').when(col('room_charge')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative room_charge values.

In [ ]:
df2.filter(col('room_charge')<0).count()

## Q29. Final Business Report

Create a final report by room_type with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('room_type').agg(count('*').alias('records'),sum('room_charge').alias('total'),avg('room_charge').alias('average'),min('room_charge').alias('minimum'),max('room_charge').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project